# 📘 Assignment 04 — Advanced Agentic RAG with OpenAI APIs

**Difficulty:** Intermediate → Advanced  
**Based on:** `03-AdvenceAgenticRAG/Advance_Agentic_RAG_with_Langchain.ipynb`

---

## 🎯 Learning Objectives

By completing this assignment you will:

1. Replace a locally-hosted Qwen2.5 model with OpenAI's **cloud API** (`gpt-4o-mini`)
2. Replace local MiniLM embeddings with OpenAI **text embeddings** (`text-embedding-3-small`)
3. Understand the trade-offs between **local models** vs **API-based models**
4. Reproduce all five agent versions (V1 → V5) from the advanced notebook using the new models
5. Track and compare **token cost**, **latency**, and **answer quality**

---

## 🔄 What Changes vs the Advanced Notebook

| Component | Advanced Notebook (Local) | This Assignment (OpenAI) |
|-----------|--------------------------|---------------------------|
| LLM | `Qwen2.5-3B-Instruct` (downloaded, ~6 GB) | `gpt-4o-mini` (API call) |
| LLM Wrapper | Custom `LocalLLM(LLM)` class | ❌ Not needed — `ChatOpenAI` is LangChain-native |
| Embeddings | `HuggingFaceEmbeddings` (MiniLM, 384-dim) | `OpenAIEmbeddings` (`text-embedding-3-small`) |
| Quantization | `BitsAndBytesConfig` (4-bit) | ❌ Not needed |
| Setup time | ~10 min (download models) | ~1 min (just an API key) |
| Cost | Free (local compute) | Pay-per-token |
| Document Loading | LangChain loaders | ✅ Same — no changes |
| Vector Store | ChromaDB | ✅ Same — no changes |
| BM25 / Hybrid Search | `BM25Retriever` | ✅ Same — no changes |
| Cross-Encoder Re-ranking | `ms-marco-MiniLM-L-6-v2` | ✅ Same — stays local |
| Contextual Compression | `LLMChainExtractor` | ✅ Same call — just uses `ChatOpenAI` |
| Agents V1–V5 | LangChain ReAct | ✅ Same structure |

> **Key insight:** Swapping the LLM and embeddings barely changes the _structure_ of the code.
> LangChain's abstraction means the rest of the pipeline is model-agnostic.

---
## ⚙️ Installation

Notice the difference from the advanced notebook:  
- ✅ Add `langchain-openai` and `openai`  
- ❌ Remove `bitsandbytes`, `accelerate`, and `transformers` (no local model needed)
- ✅ Keep everything else

In [ ]:
# Run this cell once to install all required packages
!pip install langchain langchain-community langchain-openai
!pip install openai
!pip install chromadb
!pip install sentence-transformers   # still needed for the cross-encoder (V4)
!pip install rank-bm25
!pip install pypdf beautifulsoup4
!pip install python-dotenv           # recommended: load API key from .env file

print("✅ Installation complete")

---
## 1 — Import Required Libraries

In [ ]:
# Standard library
import os
import warnings
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 1 of 7 — Add the OpenAI import                       ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# 💡 HINT: The OpenAI classes live in the `langchain_openai` package.
#          You will need two classes — one for the chat model and one for
#          embeddings. Their names follow the pattern:
#          Chat____  and  ____Embeddings
#
# add your code here


# LangChain — document loading (unchanged from advanced notebook)
from langchain_community.document_loaders import PyPDFLoader, TextLoader, BSHTMLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# LangChain — vector store & retrieval (unchanged)
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# LangChain — agent framework (unchanged)
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import Tool
from langchain.prompts import PromptTemplate

# Cross-encoder re-ranking stays local (no API needed)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

warnings.filterwarnings("ignore")
print("✅ Libraries imported")

---
## 2 — OpenAI API Setup

This section replaces the entire model-download block from the advanced notebook  
(quantization config, tokenizer, HuggingFace pipeline, LocalLLM wrapper — all gone).  
Instead you just need an API key and two constructor calls.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 2 of 7 — Load your API key                           ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# 💡 HINT: Never hard-code API keys in notebooks.
#          Two safe options:
#          a) Load from a .env file using `load_dotenv()` then `os.getenv()`
#          b) Read directly from the environment: os.environ["OPENAI_API_KEY"]
#
#          Your .env file should contain:  OPENAI_API_KEY=sk-...
#          Add .env to your .gitignore — never commit it.
#
# add your code here


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 3 of 7 — Initialise the OpenAI chat model            ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# 💡 HINT: Use ChatOpenAI (which you imported in Task 1).
#          Parameters to configure:
#            model  → "gpt-4o-mini" is cheap and fast; "gpt-4o" is more capable
#            temperature → 0 gives deterministic answers (good for RAG)
#            openai_api_key → pass the key you loaded above
#
#          Unlike the advanced notebook, you do NOT need a wrapper class.
#          ChatOpenAI already implements the LangChain LLM interface.
#
# llm = ...
# add your code here


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 4 of 7 — Initialise OpenAI Embeddings                ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# 💡 HINT: Use OpenAIEmbeddings (which you imported in Task 1).
#          Parameters to configure:
#            model → "text-embedding-3-small" (1536-dim, fast and cheap)
#                    "text-embedding-3-large" (3072-dim, more accurate)
#            openai_api_key → same key as above
#
#          The variable name must stay `embeddings` — the rest of the
#          notebook uses it without modification.
#
# embeddings = ...
# add your code here


# Quick sanity checks — run these after you fill in the code above
print(f"LLM type   : {type(llm).__name__}")
print(f"Embeddings : {type(embeddings).__name__}")
test_vec = embeddings.embed_query("hello world")
print(f"Embedding dims: {len(test_vec)}")

---
## 3 — Document Loading

> **✅ This section is identical to the Advanced RAG notebook.**  
> Copy the three document-loading functions from:  
> `Notebooks/03-AdvenceAgenticRAG/Advance_Agentic_RAG_with_Langchain.ipynb`  
> — Sections **5, 6, 7, and 8** (PDF, Text, HTML loaders + combine step).

Nothing changes here. The LangChain document loaders are model-agnostic.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 5: Load and Chunk PDF Files
# ─────────────────────────────────────────────────────────────────────────
# Paste: extract_pdf_metadata() and load_and_chunk_pdfs() here

# add your code here

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 6: Load and Chunk Text Files
# ─────────────────────────────────────────────────────────────────────────
# Paste: extract_text_metadata() and load_and_chunk_text_files() here

# add your code here

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 7: Load and Chunk HTML Files
# ─────────────────────────────────────────────────────────────────────────
# Paste: extract_html_metadata() and load_and_chunk_html_files() here

# add your code here

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 8: Combine All Document Chunks
# ─────────────────────────────────────────────────────────────────────────
# Paste the combining + stats display block here

# add your code here

---
## 4 — Create ChromaDB Vector Store

> **✅ This section is nearly identical to the Advanced RAG notebook.**  
> Copy Section 9 from the advanced notebook. The only difference is the
> `embeddings` variable — which now points to `OpenAIEmbeddings` instead
> of `HuggingFaceEmbeddings`. Because you named it `embeddings` in Task 4,
> no other change is required.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 9: Create ChromaDB Vector Store
# ─────────────────────────────────────────────────────────────────────────
# The `embeddings` variable already points to OpenAIEmbeddings.
# No other change needed.

# add your code here

---
## 5 — Create Base Retriever Tool

> **✅ Identical to the Advanced RAG notebook — copy Section 10.**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 10: Create Retriever Tool
# ─────────────────────────────────────────────────────────────────────────
# Creates: retriever + retriever_tool

# add your code here

---
## 6 — Agent V1: Basic Retriever Agent

### What changes?
In the advanced notebook, building the agent looked like this:
```python
# Advanced notebook (local)
agent = create_react_agent(llm, tools, prompt)
```
In this assignment it is... exactly the same line. `ChatOpenAI` slots in  
because it implements the same LangChain interface that `LocalLLM` did.

The ReAct prompt template and `AgentExecutor` configuration are also unchanged.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 5 of 7 — Build Agent V1                              ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# 💡 HINT: Copy the ReAct prompt template and agent creation block from
#          Section 11 of the advanced notebook.
#
#          The ONE thing you must verify: the `llm` variable passed to
#          create_react_agent() is the ChatOpenAI instance you created
#          in Task 3 — not a local pipeline.
#
#          Confirm that verbose=True is set on AgentExecutor so you can
#          watch the Thought → Action → Observation loop in the output.
#
# add your code here

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 12: Test Agent V1
# ─────────────────────────────────────────────────────────────────────────
# Use the same 3 test prompts as the advanced notebook so you can
# compare results side-by-side.

# add your code here

---
## 7 — Agent V2: Hypothetical Question Generation

### What changes?
In the advanced notebook, `generate_hypothetical_questions()` called the local
HuggingFace pipeline directly. With `ChatOpenAI`, the invocation style is different.

```python
# Advanced notebook — local pipeline call (do NOT copy this)
output = pipeline(prompt_text, max_new_tokens=150, ...)
response = output[0]["generated_text"]
```

```python
# OpenAI equivalent — use the LangChain interface (this is the pattern)
response = llm.invoke("your prompt here")
# response.content  ← this is the string you want
```

Everything else in V2 (the retriever tool wrapper, agent creation, executor) is the same.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 6 of 7 — Adapt generate_hypothetical_questions()     ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# 💡 HINT: Copy generate_hypothetical_questions() from Section 13 of the
#          advanced notebook, then replace the pipeline call with:
#
#              response = llm.invoke(prompt)
#              text = response.content          # ← extract the string
#
#          The parsing logic (splitting on newlines, stripping bullets)
#          can stay exactly the same.
#
# add your code here


# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 13 (rest of V2 setup)
# ─────────────────────────────────────────────────────────────────────────
# After adapting the function above, copy:
#   - retriever_with_hypotheticals_tool
#   - agent_v2 and executor_v2

# add your code here

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 14: Test Agent V2
# ─────────────────────────────────────────────────────────────────────────

# add your code here

---
## 8 — Agent V3: Hybrid Search (BM25 + Vector)

> **✅ This section is identical to the Advanced RAG notebook.**  
> BM25 works on raw text — it has nothing to do with the LLM or embeddings.
> Copy Section 15 and 16 as-is.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Sections 15 & 16: Hybrid Search V3
# ─────────────────────────────────────────────────────────────────────────
# Creates: BM25Retriever + EnsembleRetriever + agent_v3 + executor_v3

# add your code here

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 16: Test Agent V3
# ─────────────────────────────────────────────────────────────────────────

# add your code here

---
## 9 — Cross-Encoder Re-Ranking Setup

> **✅ This section is identical to the Advanced RAG notebook.**  
> The `ms-marco-MiniLM-L-6-v2` cross-encoder runs locally — no API needed.  
> Copy Section 17 as-is.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 17: Load Cross-Encoder
# ─────────────────────────────────────────────────────────────────────────
# Creates: cross_encoder_tokenizer, cross_encoder_model, rerank_documents()

# add your code here

---
## 10 — Agent V4: Hybrid Search + Re-Ranking

> **✅ This section is identical to the Advanced RAG notebook.**  
> Copy Sections 18 and 19 as-is.  
> The `rerank_documents()` function you copied above is called inside the
> retriever wrapper — nothing touches the LLM at this stage.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Sections 18 & 19: Re-Ranking V4
# ─────────────────────────────────────────────────────────────────────────
# Creates: reranking_tool + agent_v4 + executor_v4

# add your code here

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 19: Test Agent V4
# ─────────────────────────────────────────────────────────────────────────

# add your code here

---
## 11 — Contextual Compression Setup (V5)

### What changes?
`LLMChainExtractor.from_llm(llm)` is the same call as in the advanced notebook.  
The difference is what `llm` points to.

With a local 3B model, compression was unreliable — the model often returned  
garbled text or ignored the instruction to extract only the relevant part.  

With `gpt-4o-mini`, compression becomes a key strength: the model reliably  
strips irrelevant content and returns only the useful fragment.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 7 of 7 — Set up Contextual Compression               ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# 💡 HINT: Copy the LLMChainExtractor block from Section 20 of the
#          advanced notebook. The call is identical:
#
#              compressor = LLMChainExtractor.from_llm(???)
#
#          Replace the argument with the ChatOpenAI `llm` you created
#          in Task 3. That is the only change.
#
#          After you run this cell, compare the compression ratio:
#          does GPT-4o-mini compress more accurately than the local model?
#
# add your code here

---
## 12 — Agent V5: Full Advanced Pipeline

> **✅ Mostly identical to the Advanced RAG notebook.**  
> Copy Sections 21 and 22 as-is.  
> The `compressor` variable (which you just created above) slots in
> directly — no changes to the agent structure.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Sections 21 & 22: Full Pipeline V5
# ─────────────────────────────────────────────────────────────────────────
# Creates: advanced_tool + agent_v5 + executor_v5

# add your code here

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 📋 COPY FROM ADVANCED NOTEBOOK — Section 22: Test Agent V5
# ─────────────────────────────────────────────────────────────────────────

# add your code here

---
## 13 — Reflection: Local vs Cloud

Now that you have run both notebooks, answer these questions in this cell:

| Question | Your observation |
|----------|------------------|
| Which was faster per query — local or API? | |
| Did answer quality improve with GPT-4o-mini? | |
| Was contextual compression (V5) more reliable with GPT-4o-mini? | |
| What happens if the OpenAI API is unavailable? | |
| For a privacy-sensitive use case, which would you choose? | |

---

## 🏆 Bonus Challenges

If you finish early, try these extensions:

**Bonus 1 — Token cost tracking**  
LangChain provides a `get_openai_callback()` context manager.  
Wrap each agent call to log `total_tokens`, `prompt_tokens`, `completion_tokens`, and `total_cost`.
```python
from langchain_community.callbacks import get_openai_callback
with get_openai_callback() as cb:
    result = executor_v5.invoke({"input": question})
    print(cb)  # hint: explore what cb exposes
```

**Bonus 2 — Model comparison**  
Run the same query through `gpt-4o-mini` and `gpt-4o`.  
Compare: answer quality, token usage, and cost.

**Bonus 3 — OpenAI re-ranking**  
In the advanced notebook, V4 uses a local cross-encoder for re-ranking.  
Can you replace it with a GPT-4o-mini call that scores document relevance?  
When would you prefer each approach?